# Session 3 — What an explanation says about a model, and what it does not

**Learning objectives:** distinguish what a SHAP explanation says about a model from what it says about a fraud mechanism (objective 4), and judge how easily a screen can be evaded. About 100 minutes. The session ends with a one-page critique.

In [ ]:
# Setup: works locally (run from the teaching/ folder) and in Google Colab.
import os, sys, subprocess
from pathlib import Path
if 'google.colab' in sys.modules and not Path('rugpull-static-detection').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/fazaaf/rugpull-static-detection.git'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost==3.4.1', 'shap==0.54.1', 'scikit-learn==1.6.1', 'openpyxl'], check=True)
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path('rugpull-static-detection')] if (p / 'data' / 'D_single_era_matched.csv').exists())
print('repository root:', ROOT.resolve())

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import matthews_corrcoef
from sklearn.utils.class_weight import compute_sample_weight

D = pd.read_csv(ROOT / 'data' / 'D_single_era_matched.csv'); D['address'] = D.address.str.lower()
G = pd.read_csv(ROOT / 'data' / 'groups.csv')
D = D.merge(G.drop(columns=['label', 'year', 'solc_minor']), on='address', validate='1:1')
FEAT = list(D.loc[:, 'total_vulnerabilities':'code_length'].columns)   # the 29 features
X, y = D[FEAT], D.label.values
print(len(D), 'contracts |', y.sum(), 'rug pulls |', (y == 0).sum(), 'legitimate |', len(FEAT), 'features')

def model(seed=0):
    # the settings used in the paper (no tuning)
    return xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                             eval_metric='logloss', random_state=seed, n_jobs=4)

def fit(m, Xtr, ytr):
    return m.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))

In [ ]:
import shap, matplotlib.pyplot as plt
# Out-of-fold predictions and SHAP values (feature-space grouping, one repetition): every contract is explained by a model that did not see it.
p = np.zeros(len(D)); sv = np.zeros((len(D), len(FEAT))); base = np.zeros(len(D)); fold = np.zeros(len(D), int)
for k, (tr, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, y, D.feature_cluster.values)):
    m = fit(model(0), X.iloc[tr], y[tr]); p[te] = m.predict_proba(X.iloc[te])[:, 1]
    e = shap.TreeExplainer(m)(X.iloc[te]); sv[te] = e.values; base[te] = e.base_values; fold[te] = k
imp = pd.Series(np.abs(sv).mean(0), index=FEAT).sort_values(ascending=False)
imp.head(10).iloc[::-1].plot.barh(figsize=(6, 4), title='Mean |SHAP| (log-odds), top 10'); plt.tight_layout(); plt.show()

## Part A — Read the global explanation (20 min)

**Question A1.** `optimization` counts Slither findings such as *constable-states* ("this state variable could be declared constant"). Is that a fraud mechanism? Why might it still separate the classes in this data?

**Question A2.** Which of the twelve keyword indicators appear in the top 10? Recall Session 2: what do you need to know before you interpret a small SHAP value for `has_blacklist`?

In [ ]:
# Two individual explanations: the highest-scoring legitimate token and the lowest-scoring rug pull
i_fp = int(np.argmax(np.where(y == 0, p, -1)))
i_fn = int(np.argmin(np.where(y == 1, p, 2)))
for i, title in [(i_fp, 'Legitimate token with the highest rug pull score'), (i_fn, 'Rug pull with the lowest score')]:
    print(title, '|', D.address[i], '| score', round(p[i], 3))
    shap.plots.waterfall(shap.Explanation(sv[i], base[i], X.iloc[i].values, feature_names=FEAT), max_display=10, show=False)
    plt.title(title); plt.tight_layout(); plt.show()

**Question A3.** Look up the flagged legitimate token in `audit/top20_legit_audit.xlsx` (most of the top-20 tokens are there). What did the audit find? Does the explanation tell you *why the token is risky*, or *which known contracts it resembles*?

## Part B — Evasion test (25 min)

A contract author can remove optimization findings by declaring variables `constant` or `immutable`, and can change the structural metrics by adding harmless functions. Neither touches the mechanism that traps buyers.

In [ ]:
legmed = D.loc[y == 0, FEAT].median()
def perturb(Xt, scenario):
    Z = Xt.copy()
    if scenario in ('S1', 'S4'):
        Z['total_vulnerabilities'] = Z.total_vulnerabilities - Z.optimization; Z['optimization'] = 0
    if scenario in ('S3', 'S4'):
        for f in ['func_count', 'modifier_count', 'event_count', 'require_count', 'code_length']: Z[f] = legmed[f]
    return Z
rec = {s: [] for s in ['S0', 'S1', 'S3', 'S4']}
for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, y, D.feature_cluster.values):
    m = fit(model(0), X.iloc[tr], y[tr]); te_r = te[y[te] == 1]
    for s in rec: rec[s].append((m.predict_proba(perturb(X.iloc[te_r], s))[:, 1] >= 0.5).mean())
pd.Series({'S0 unchanged': np.mean(rec['S0']), 'S1 no optimization findings': np.mean(rec['S1']),
           'S3 structure = legitimate median': np.mean(rec['S3']), 'S4 = S1 + S3': np.mean(rec['S4'])}, name='rug pull recall').round(3)

**Question B1.** The paper reports recall 0.961 → 0.357 (S1) → 0.070 (S4). What does this imply for a tool that publishes its feature list?

## Part C — One-page critique (35 min, individually)

Read the claim below and write a critique of at most one page. Use at least one result from each session (splitting, indicator validity, explanation or evasion).

> *"Our explainable XGBoost model detects rug pull tokens from their source code with an MCC of 0.84. SHAP shows which code features reveal fraud, so investors can use it to screen new tokens before buying."*

Points to address: what the 0.84 measures; whether "new tokens" are covered by the evaluation; what the SHAP values explain; whether the features reveal fraud mechanisms; what an adversary would do.